# Model Evaluation & Business Performance Analysis

## Overview  
This notebook focuses on the final evaluation of the trained machine learning models for the Bank Marketing dataset. After completing preprocessing, model training, and segmentation in previous stages, we now assess model performance on the held-out test set to obtain an unbiased estimate of real-world effectiveness.

The evaluation is conducted from both a predictive and a business-oriented perspective. In addition to standard classification metrics, we incorporate a cost–benefit framework that reflects the economic impact of marketing decisions, allowing us to evaluate models in terms of expected profit rather than accuracy alone.

We compare a global modeling strategy (one model for all clients) against a segmented strategy (cluster-specific models), both at the overall level and within individual clusters.

## Objectives  
- Evaluate the final global model on the test set using calibrated probabilities  
- Evaluate the segmented (cluster-specific) modeling approach on the test set  
- Compare global vs segmented strategies using both predictive and business metrics  
- Compute expected profit using a cost–benefit decision framework  
- Apply optimal decision thresholds derived from validation data  
- Analyze performance differences across clusters to assess segmentation impact  
- Provide a final comparison of strategies to support decision-making  

## Dataset Description  
This evaluation uses the held-out test dataset generated in the preprocessing phase. The dataset contains engineered features, including categorical encodings, binary indicators, and interaction variables, as well as a cluster assignment for each observation.

The test set is not used during model training or threshold selection, ensuring that all reported results reflect unbiased out-of-sample performance.

The models evaluated include:
- A global model trained on the full dataset  
- Multiple cluster-specific models trained independently per segment  

Each model outputs calibrated probabilities, which are then transformed into binary decisions using pre-selected thresholds.

## Key Considerations  
- **Out-of-sample evaluation**: All results are computed on the test set to ensure unbiased performance estimates  
- **Consistency with training pipeline**: Feature alignment, calibration, and thresholds are applied exactly as defined during training  
- **Business-oriented evaluation**: Performance is assessed not only using classification metrics but also expected profit  
- **Threshold-based decision rule**: Final predictions are derived using thresholds optimized on validation data  
- **Segmentation consistency**: Cluster-specific models are evaluated only within their respective segments to ensure fair comparison  
- **Comparability of strategies**: Global and segmented approaches are evaluated under the same cost–benefit assumptions  

## Outcome  
By the end of this notebook, we obtain a comprehensive evaluation of the modeling approaches, including:

- Final performance metrics for the global model  
- Final performance metrics for the segmented (cluster-based) model  
- Cluster-level comparisons between global and segmented strategies  
- A consolidated view of expected profit and operational impact  
- A final assessment of whether segmentation provides measurable business value  

These results provide the basis for selecting the final deployment strategy and understanding the trade-offs between model simplicity and segment-level specialization.

In [1]:
import numpy as np
import pandas as pd
import joblib
from pathlib import Path
from sklearn.metrics import (
    average_precision_score,
    precision_score,
    recall_score,
    f1_score,
)

from utils.utils import load_dataset, evaluate_with_profit

MODEL_DIR = Path("models")

In [2]:
# quarto preview 04_evaluation.ipynb --to pdf
# quarto render 04_evaluation.ipynb
# black 04_evaluation.ipynb

In [3]:
# Global artifacts
global_model = joblib.load(MODEL_DIR / "global_model.joblib")
global_calibrator = joblib.load(MODEL_DIR / "global_calibrator.joblib")
global_metadata = joblib.load(MODEL_DIR / "global_metadata.joblib")
global_threshold = joblib.load(MODEL_DIR / "global_threshold.joblib")

In [4]:
# Cluster artifacts
cluster_models = {}
cluster_calibrators = {}
cluster_metadata = {}

cluster_thresholds = joblib.load(MODEL_DIR / "cluster_thresholds.joblib")

for cluster_id in cluster_thresholds.keys():
    cluster_models[cluster_id] = joblib.load(MODEL_DIR / f"c{cluster_id}_model.joblib")
    cluster_calibrators[cluster_id] = joblib.load(
        MODEL_DIR / f"c{cluster_id}_calibrator.joblib"
    )
    cluster_metadata[cluster_id] = joblib.load(
        MODEL_DIR / f"c{cluster_id}_metadata.joblib"
    )

In [5]:
X_test_full, y_test = load_dataset("02", "test")

cluster_col = "cluster"

# Keep cluster labels for segmentation
X_test = X_test_full.drop(columns=[cluster_col])


test
X shape: (9043, 60)
y shape: (9043,)

X head:


,cat__job_admin.,cat__job_blue-collar,cat__job_entrepreneur,cat__job_housemaid,cat__job_management,cat__job_retired,cat__job_self-employed,cat__job_services,cat__job_student,cat__job_technician,...,bin__housing,bin__loan,bin__poutcome_success,bin__contact_is_unknown,bin__poutcome_is_unknown,bin__multiple_contacts_current_campaign,bin__previous_and_many_current_contacts,bin__many_contacts_and_housing_loan,bin__both_housing_and_personal_loan,cluster
0,0.0,0.0,0.0,0.0,1.0,0.0,0.0,0.0,0.0,0.0,...,0.0,0.0,0.0,0.0,1.0,0.0,0.0,0.0,0.0,3
1,0.0,1.0,0.0,0.0,0.0,0.0,0.0,0.0,0.0,0.0,...,0.0,0.0,0.0,0.0,0.0,0.0,0.0,0.0,0.0,2
2,0.0,0.0,0.0,0.0,0.0,1.0,0.0,0.0,0.0,0.0,...,0.0,0.0,1.0,0.0,0.0,1.0,1.0,0.0,0.0,0


In [6]:
C_call = 1.0
B_sub = 10.0

## 1. Overall Comparison

### 1. Global Model Evaluation

In [7]:
# Align features
X_test_global = X_test[global_metadata["features"]]

# Predict probabilities (raw + calibrated)
global_probs_raw = global_model.predict_proba(X_test_global)[:, 1]
global_probs_cal = global_calibrator.transform(global_probs_raw)

In [8]:
global_metrics = evaluate_with_profit(
    y_true=y_test,
    y_prob=global_probs_cal,
    threshold=global_threshold,
    C_call=C_call,
    B_sub=B_sub,
)

global_results_df = pd.DataFrame([global_metrics], index=["Global"])
display(global_results_df)

,AUC_PR,Precision,Recall,F1,% Contacted,Expected Profit
Global,0.419732,0.264336,0.714556,0.385911,31.626673,5158.96754


The question that is answered by this is: if I use ONE model for everyone, how do I perform?

### 2. Segmented Model Evaluation

We apply a different model per cluster, then recombine results

In [9]:
all_probs_cluster = np.zeros(len(y_test))
y_pred_segmented = np.zeros(len(y_test))

In [10]:
for cluster_id in cluster_thresholds.keys():

    mask = X_test_full[cluster_col] == cluster_id

    X_k = X_test_full[mask].drop(columns=[cluster_col])
    y_k = y_test[mask]

    # align features
    X_k = X_k[cluster_metadata[cluster_id]["features"]]

    model_k = cluster_models[cluster_id]
    calibrator_k = cluster_calibrators[cluster_id]
    threshold_k = cluster_thresholds[cluster_id]

    probs_k = calibrator_k.transform(model_k.predict_proba(X_k)[:, 1])

    all_probs_cluster[mask] = probs_k
    y_pred_segmented[mask] = (probs_k >= threshold_k).astype(int)

In [11]:
# evaluate segmented model
segmented_metrics = {
    "AUC_PR": average_precision_score(y_test, all_probs_cluster),
    "Precision": precision_score(y_test, y_pred_segmented, zero_division=0),
    "Recall": recall_score(y_test, y_pred_segmented),
    "F1": f1_score(y_test, y_pred_segmented),
    "% Contacted": y_pred_segmented.mean() * 100,
    "Expected Profit": np.sum(
        all_probs_cluster[y_pred_segmented == 1] * B_sub - C_call
    ),
}

segmented_results_df = pd.DataFrame([segmented_metrics], index=["Segmented"])
display(segmented_results_df)

,AUC_PR,Precision,Recall,F1,% Contacted,Expected Profit
Segmented,0.398141,0.272295,0.649338,0.383692,27.900033,4833.213341


The question that is answered by this is: if I specialize by cluster, how do I perform overall?

### 3. Comparison

In [12]:
final_comparison = pd.concat([global_results_df, segmented_results_df])

final_comparison["Expected Profit"] = final_comparison["Expected Profit"].round(0)
final_comparison["% Contacted"] = final_comparison["% Contacted"].round(2)

display(final_comparison)

,AUC_PR,Precision,Recall,F1,% Contacted,Expected Profit
Global,0.419732,0.264336,0.714556,0.385911,31.63,5159.0
Segmented,0.398141,0.272295,0.649338,0.383692,27.90,4833.0


The question that is answered by this is: which strategy wins overall?

The global and segmented strategies show a trade-off between predictive performance and business efficiency. The global model achieves higher recall and slightly better AUC-PR, indicating stronger overall ability to capture positive cases across the full population. In contrast, the segmented approach improves precision while reducing recall, reflecting a more conservative targeting strategy.

From a business perspective, the global model results in higher expected profit under the chosen decision rule, although it also leads to a higher percentage of contacted clients. The segmented strategy reduces contact volume but does not fully translate this reduction into higher profitability at the aggregate level.

Overall, both approaches exhibit similar F1 performance, suggesting comparable balance between precision and recall, but differ primarily in how they trade off coverage versus efficiency.

## 2. Per-cluster comparison

In [13]:
comparison_by_cluster = []

for cluster_id in sorted(cluster_thresholds.keys()):

    mask = X_test_full[cluster_col] == cluster_id
    y_k = y_test[mask]

    # =========================
    # GLOBAL MODEL
    # =========================
    probs_g = global_probs_cal[mask]

    metrics_g = evaluate_with_profit(
        y_true=y_k,
        y_prob=probs_g,
        threshold=global_threshold,
        C_call=C_call,
        B_sub=B_sub,
    )

    comparison_by_cluster.append(
        {"Cluster": cluster_id, "Model": "Global", **metrics_g}
    )

    # =========================
    # SEGMENTED MODEL
    # =========================
    probs_k = all_probs_cluster[mask]

    metrics_k = evaluate_with_profit(
        y_true=y_k,
        y_prob=probs_k,
        threshold=cluster_thresholds[cluster_id],
        C_call=C_call,
        B_sub=B_sub,
    )

    comparison_by_cluster.append(
        {"Cluster": cluster_id, "Model": "Segmented", **metrics_k}
    )

In [14]:
comparison_by_cluster_df = pd.DataFrame(comparison_by_cluster)

comparison_by_cluster_df["Expected Profit"] = comparison_by_cluster_df[
    "Expected Profit"
].round(0)
comparison_by_cluster_df["% Contacted"] = comparison_by_cluster_df["% Contacted"].round(
    2
)

perf_metrics = ["Cluster", "Model", "AUC_PR", "Precision", "Recall", "F1"]
display(comparison_by_cluster_df[perf_metrics])

,Cluster,Model,AUC_PR,Precision,Recall,F1
0,0,Global,0.478119,0.315141,0.788546,0.450314
1,0,Segmented,0.443843,0.286416,0.770925,0.417661
2,1,Global,0.357666,0.267465,0.582609,0.366621
3,1,Segmented,0.307271,0.292035,0.430435,0.347979
4,2,Global,0.412600,0.234957,0.666667,0.347458
5,2,Segmented,0.382807,0.272401,0.617886,0.378109
6,3,Global,0.428512,0.255261,0.785915,0.385359
7,3,Segmented,0.414652,0.257143,0.735211,0.381022


The per-cluster results highlight meaningful heterogeneity in how the global and segmented strategies perform across different customer segments.

In all clusters, the global model tends to achieve higher recall, indicating stronger ability to identify positive cases, while the segmented models generally improve precision, reflecting more conservative and targeted decision boundaries within each segment.

Performance differences are not uniform across clusters. Some segments show only marginal differences between approaches, while others exhibit more pronounced trade-offs between recall and precision depending on whether a global or specialized model is used. This reinforces the idea that customer behavior and model effectiveness vary across clusters, making segmentation relevant at the local level even if gains are not consistent across all segments.

In [15]:
final_cluster_comparison = comparison_by_cluster_df.reset_index().pivot(
    index="Cluster", columns="Model", values=["Expected Profit", "% Contacted"]
)

# flatten columns
final_cluster_comparison.columns = [
    f"{col[1]} {col[0]}" for col in final_cluster_comparison.columns
]

final_cluster_comparison = final_cluster_comparison[
    [
        "Global Expected Profit",
        "Segmented Expected Profit",
        "Global % Contacted",
        "Segmented % Contacted",
    ]
]

display(final_cluster_comparison)

,Global Expected Profit,Segmented Expected Profit,Global % Contacted,Segmented % Contacted
Cluster,,,,
0,1313.0,1319.0,35.24,37.90
1,805.0,558.0,18.09,12.24
2,1125.0,844.0,31.71,25.35
3,1916.0,2112.0,44.41,41.24


The question that is answered by this is: does segmentation actually improve results WHERE it matters? 

At the cluster level, the segmented strategy shows mixed performance relative to the global model, with clear variation across segments.

In some clusters, segmentation improves expected profit, suggesting that specialized models are better aligned with local behavioral patterns and decision boundaries (Cluster 0 and 3). However, in other clusters, the global model performs better (Cluster 1 and 4), indicating that pooling information across the full population can still be beneficial when segment-specific data is more limited or less stable.

The percentage of contacted clients also varies between strategies, reflecting different threshold behaviors per cluster. This leads to differences in operational intensity that are not uniform across segments.

Overall, the results suggest that the value of segmentation is cluster-dependent rather than globally consistent, with some segments benefiting from specialization while others favor a unified modeling approach.